# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 26, 2026  
**Hardware:** Optimized for EC2 instances  
**Model Strategy:** Single Combined model for all cohorts

## Overview

This notebook provides an interactive workflow for:

1. **Training Calculator Models** - Train CatBoost, XGBoost, and XGBoost RF models using the **Combined** cohort (single model for all patients)
2. **SHAP + FFA Analysis** - Generate causal factors and dashboard data using SHAP values and XGBoost rule extraction
3. **Results Inspection** - View top causal factors, feature importance, and model performance

## Model Architecture

- **Single Model Approach**: One Combined model is trained for all cohorts (CHD, Cardiomyopathy, Myocarditis)
- **Primary Diagnosis Feature**: `primary_etiology` is included to distinguish between etiologies
- **Feature Engineering**: Automatic derivation of combined variables (VAD, Ventilation, ECMO, donor ratios)

## Workflow Steps

- **Step 1:** Train Combined model (single model for all cohorts)
- **Step 2:** Run SHAP/FFA analysis to extract causal factors
- **Step 3:** Inspect results and export dashboard data

## Expected Runtime

- **Model Training (Combined):** ~15-30 minutes
- **SHAP/FFA Analysis (Combined):** ~10-20 minutes
- **Total:** ~30-50 minutes on EC2


## 1. Input Features Overview

### Required Input Variables for Risk Calculator

The model uses the following input features, with automatic feature engineering for derived variables:

#### Primary Diagnosis & History
- **Primary Diagnosis** (`primary_etiology`) - Congenital Heart Disease, Cardiomyopathy, Myocarditis, Other
- **Previous Cardiac Surgery** (`hxsurg`) - History of surgery (Yes/No)
- **Laterality Disorder** (`chd_lat`) - Composite variable (Yes/No)
  - Derived from: `chd_dex`, `chd_si`, `chd_heter`, `chd_iivc`, `chd_bivc`, `chd_lsvc`, `chd_raa`, `chd_avd`

#### Cardiac Support Devices (Combined Variables)
- **ECMO** (`ecmo_combined`) - ECMO at transplant OR listing
  - Derived from: `txecmo` OR `slecmo`
- **VAD** (`vad_combined`) - VAD at transplant OR listing
  - Derived from: `txvad` OR `slvad`
- **Mechanical Ventilation** (`vent_combined`) - Ventilation at transplant OR listing
  - Derived from: `txvent` OR `slvent` OR `ltxtrach` OR `hxtrach`

#### Demographics & Age
- **Age at Transplant** (`age_txpl`) - Years (priority over `age_listing`)
- **Age at Listing** (`age_listing`) - Years (fallback)

#### Renal Function
- **Dialysis History** (`hxdysdia` / `hxdysdia_bin`) - History of dialysis (ever)
- **eGFR at Transplant** (`egfr_tx`) - Calculated from height and creatinine
  - Formula: `egfr_tx = 0.413 × height_txpl / txcreat_r`
- **eGFR at Listing** (`egfr_listing`) - Calculated from height and creatinine

#### Liver Function
- **ALT at Transplant** (`txalt`) - U/L (priority over `lsalt`)
- **AST at Transplant** (`txast`) - U/L (priority over `lsast`)
- **Direct Bilirubin at Transplant** (`txbili_d_r`) - mg/dL (priority over `lsbili_d_r`)
- **Total Bilirubin at Transplant** (`txbili_t_r`) - mg/dL (priority over `lsbili_t_r`)

#### Nutrition
- **Serum Albumin at Transplant** (`txsa_r`) - g/dL (priority over `lssab_r`)
- **Total Protein at Transplant** (`txtp_r`) - g/dL (priority over `lstp_r`)

#### Immunology
- **cPRA at Transplant** (`txfcpra`) - Flow cytometry PRA % (priority over `lsfcpra`)
- **cPRA at Listing** (`lsfcpra`) - Flow cytometry PRA % (fallback)

#### Donor Characteristics
- **Donor Ischemic Time** (`donisch`) - Minutes (default: < 240 minutes if not provided)
- **Donor/Recipient Weight Ratio** (`donor_weight_ratio`) - Percentage
  - Formula: `(weight_donor / weight_txpl) × 100`
  - Model assumption: 70-200%
- **Donor/Recipient Size Ratio** (`donor_size_ratio`) - Percentage
  - Formula: `(height_donor / height_txpl) × 100`
  - Model assumption: 70-200%

### Additional Features

The model also includes:
- All CHD subtype variables (40+ subtypes, e.g., `chd_hlh`, `chd_lsvc`, `chd_si`, etc.)
- Additional lab values and clinical history variables
- Derived categorical variables (eGFR categories, high/low indicators)
- Donor characteristics and transplant details

### Feature Engineering

The following variables are automatically created during training and inference:
1. `ecmo_combined` - ECMO combined
2. `vad_combined` - VAD combined
3. `vent_combined` - Ventilation combined
4. `donor_weight_ratio` - Donor/recipient weight ratio
5. `donor_size_ratio` - Donor/recipient height ratio
6. `chd_lat` - Laterality disorder composite
7. `egfr_tx` - eGFR at transplant (if not provided, calculated from height/creatinine)
8. `egfr_listing` - eGFR at listing
9. `egfr_tx_cat` - eGFR category (severe/moderate/mild/normal)
10. `egfr_listing_cat` - eGFR category at listing
11. Additional derived variables (BMI, high/low indicators, etc.)

---

## 2. Setup and Configuration

Load required packages and configure paths.

In [ ]:
import sys
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add project paths
PROJECT_ROOT = Path().resolve().parent.parent.parent
CALCULATOR_DIR = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(CALCULATOR_DIR))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("=" * 80)
print("PHTS Calculator Workflow")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calculator directory: {CALCULATOR_DIR}")
print("=" * 80)

In [ ]:
# Configuration
DEBUG_MODE = False  # Set to True for quick testing (fewer splits)

# Model Strategy: Single Combined model for all cohorts
# Note: The training script will always train Combined model regardless of --cohort argument
COHORT = "Combined"  # Single model approach - Combined model for all cohorts

# SHAP/FFA configuration
TOP_K = 10  # Number of top causal factors to extract
WEIGHT_CATBOOST = 0.6  # Weight for CatBoost importance
WEIGHT_XGBOOST = 0.4  # Weight for XGBoost importance

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
print(f"  Model Strategy: Single Combined model (for all cohorts)")
print(f"  Cohort: {COHORT}")
print(f"  Top K factors: {TOP_K}")
print(f"  CatBoost weight: {WEIGHT_CATBOOST}")
print(f"  XGBoost weight: {WEIGHT_XGBOOST}")
print(f"\nNote: The model includes primary_etiology to distinguish between:")
print(f"  - Congenital Heart Disease")
print(f"  - Cardiomyopathy")
print(f"  - Myocarditis")
print(f"  - Other")

In [ ]:
# Check dependencies
print("\nChecking dependencies...")

try:
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor
    import xgboost as xgb
    import shap
    print("✓ All required packages are installed")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  XGBoost: {xgb.__version__}")
    print(f"  SHAP: {shap.__version__}")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Please install: pip install numpy pandas catboost xgboost shap")

In [ ]:
# Check data availability
print("\nChecking data availability...")

data_file = PROJECT_ROOT / "graft-loss" / "data" / "phts_txpl_ml.sas7bdat"
if data_file.exists():
    size_mb = data_file.stat().st_size / (1024 * 1024)
    print(f"✓ Data file found: {data_file}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"⚠ Data file not found: {data_file}")
    print("  You may need to download the data file first")

# Check calculator directory structure
outputs_dir = CALCULATOR_DIR / "outputs"
if outputs_dir.exists():
    print(f"✓ Outputs directory exists: {outputs_dir}")
else:
    print(f"✓ Creating outputs directory: {outputs_dir}")
    outputs_dir.mkdir(parents=True, exist_ok=True)

## 3. Train Calculator Models

Train CatBoost, XGBoost, and XGBoost RF models using the **Combined** cohort.

**Note:** The training script enforces a single Combined model strategy. Even if you specify a different cohort, it will train the Combined model for all patients.

In [ ]:
# Import training function
from train_python_models import train_models_for_cohort

print(f"\n{'=' * 80}")
print("Training Calculator Models")
print(f"{'=' * 80}")
print(f"Model Strategy: Single Combined model for all cohorts")
print(f"{'=' * 80}")

# Train Combined model (single model for all cohorts)
print(f"\nTraining Combined model (for all cohorts)...")
print("-" * 80)
print("Note: This model includes:")
print("  - primary_etiology feature to distinguish etiologies")
print("  - All derived variables (vad_combined, vent_combined, donor ratios, chd_lat)")
print("  - All required input features from risk calculator")
print("-" * 80)

try:
    train_models_for_cohort(COHORT)
    print(f"\n✓ Combined model training complete!")
    print(f"\nThe model is now ready for:")
    print(f"  - Risk prediction for all cohorts (CHD, Cardiomyopathy, Myocarditis)")
    print(f"  - SHAP/FFA analysis to extract causal factors")
except Exception as e:
    print(f"\n✗ Error training Combined model: {e}")
    logger.error(f"Error training Combined model", exc_info=True)

print(f"\n{'=' * 80}")
print("Model training complete!")
print(f"{'=' * 80}")

In [ ]:
# Check training results
import json

print("\nTraining Results Summary:")
print("-" * 80)

best_model_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "best_model.txt"
if best_model_file.exists():
    print(f"\n{COHORT} Model (for all cohorts):")
    with open(best_model_file, 'r') as f:
        print(f.read())
else:
    print(f"\n⚠ {COHORT}: Best model file not found")

# List model files
models_dir = CALCULATOR_DIR / "outputs" / "models" / COHORT
if models_dir.exists():
    model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
    if model_files:
        print(f"\n  Model files ({len(model_files)}):")
        for model_file in sorted(model_files):
            size_mb = model_file.stat().st_size / (1024 * 1024)
            print(f"    {model_file.name} ({size_mb:.2f} MB)")
    
    # Check feature count
    feature_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "feature_names.json"
    if feature_file.exists():
        with open(feature_file, 'r') as f:
            features = json.load(f)
            print(f"\n  Total features: {len(features)}")
            print(f"  Includes primary_etiology: {'primary_etiology' in features}")
            print(f"  Includes vad_combined: {'vad_combined' in features}")
            print(f"  Includes vent_combined: {'vent_combined' in features}")
            print(f"  Includes donor_weight_ratio: {'donor_weight_ratio' in features}")
            print(f"  Includes donor_size_ratio: {'donor_size_ratio' in features}")
            print(f"  Includes chd_lat: {'chd_lat' in features}")

## 4. Run SHAP + FFA Analysis

Generate SHAP values and extract causal factors using Formal Feature Attribution for the Combined model.

In [ ]:
# Run SHAP/FFA workflow for Combined model
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis")
print(f"{'=' * 80}")
print(f"Analyzing Combined model (for all cohorts)...")
print("-" * 80)

try:
    result = subprocess.run(
        [
            sys.executable,
            str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
            "--cohort", COHORT,
            "--top-k", str(TOP_K),
            "--weight-catboost", str(WEIGHT_CATBOOST),
            "--weight-xgboost", str(WEIGHT_XGBOOST)
        ],
        cwd=str(CALCULATOR_DIR),
        capture_output=False,  # Show output in real-time
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✓ Combined model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors")
        print(f"  - Feature importance rankings")
        print(f"  - Dashboard data for risk calculator")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

## 5. Inspect Results

View top causal factors, feature importance, and dashboard data for the Combined model.

In [ ]:
# Load and display dashboard data
import json
import pandas as pd

print("\n" + "=" * 80)
print("Results Summary - Combined Model")
print("=" * 80)

dashboard_data_file = (
    CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
)

if dashboard_data_file.exists():
    print(f"\n{COHORT} Model - Top {TOP_K} Causal Factors:")
    print("-" * 80)
    
    with open(dashboard_data_file, 'r') as f:
        dashboard_data = json.load(f)
    
    top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
    
    if top_factors:
        for idx, factor in enumerate(top_factors, 1):
            importance = factor.get('causal_responsibility', 
                                 factor.get('importance', 
                                           factor.get('combined_importance_norm', 0)))
            print(f"{idx:2d}. {factor['feature']:40s} "
                  f"(Importance: {importance:.4f})")
    else:
        print("  (No causal factors available)")
    
    # Display summary statistics
    if 'summary' in dashboard_data:
        print(f"\n  Summary Statistics:")
        summary = dashboard_data['summary']
        for key, value in summary.items():
            print(f"    {key}: {value}")
    
    # Check for key features in top factors
    print(f"\n  Key Features Check:")
    top_feature_names = [f['feature'] for f in top_factors]
    key_features = {
        'primary_etiology': any('primary_etiology' in f for f in top_feature_names),
        'vad_combined': 'vad_combined' in top_feature_names,
        'vent_combined': 'vent_combined' in top_feature_names,
        'ecmo_combined': 'ecmo_combined' in top_feature_names,
        'donor_weight_ratio': 'donor_weight_ratio' in top_feature_names,
        'donor_size_ratio': 'donor_size_ratio' in top_feature_names,
        'chd_lat': 'chd_lat' in top_feature_names,
        'egfr_tx': 'egfr_tx' in top_feature_names or any('egfr' in f for f in top_feature_names),
        'txfcpra': 'txfcpra' in top_feature_names,
        'hxsurg': 'hxsurg' in top_feature_names
    }
    for feature, present in key_features.items():
        status = "✓" if present else "○"
        print(f"    {status} {feature}")
else:
    print(f"\n⚠ Dashboard data not found")
    print(f"  Expected: {dashboard_data_file}")
    print("  Run SHAP/FFA analysis first (Section 4)")

In [ ]:
# Load and display feature importance
print("\n" + "=" * 80)
print("Feature Importance Rankings - Combined Model")
print("=" * 80)

importance_files = list(
    (CALCULATOR_DIR / "outputs" / "models" / COHORT).glob("importance_*.csv")
)

if importance_files:
    print(f"\n{COHORT} Model - Feature Importance:")
    print("-" * 80)
    
    for imp_file in sorted(importance_files):
        model_name = imp_file.stem.replace(f"importance_{COHORT}_", "")
        print(f"\n  {model_name}:")
        df = pd.read_csv(imp_file)
        print(f"    Total features: {len(df)}")
        print(f"    Top 10 features:")
        top10 = df.nlargest(10, 'importance')
        for idx, row in top10.iterrows():
            print(f"      {row['feature']:40s} {row['importance']:.4f}")
        
        # Check for key features
        feature_list = df['feature'].tolist()
        print(f"\n    Key Features Status:")
        key_features = {
            'primary_etiology': any('primary_etiology' in f for f in feature_list),
            'vad_combined': 'vad_combined' in feature_list,
            'vent_combined': 'vent_combined' in feature_list,
            'ecmo_combined': 'ecmo_combined' in feature_list,
            'donor_weight_ratio': 'donor_weight_ratio' in feature_list,
            'donor_size_ratio': 'donor_size_ratio' in feature_list,
            'chd_lat': 'chd_lat' in feature_list,
            'egfr_tx': 'egfr_tx' in feature_list,
            'txfcpra': 'txfcpra' in feature_list,
            'hxsurg': 'hxsurg' in feature_list
        }
        for feature, present in key_features.items():
            status = "✓" if present else "○"
            if present:
                rank = df[df['feature'] == feature].index[0] + 1 if feature in feature_list else "N/A"
                print(f"      {status} {feature:25s} (Rank: {rank})")
            else:
                print(f"      {status} {feature:25s} (Not found)")
else:
    print(f"\n⚠ No feature importance files found")
    print("  Train models first (Section 3)")

## 6. Visualizations (Optional)

Create visualizations of results for the Combined model.

In [ ]:
# Plot top causal factors (if matplotlib is available)
try:
    import matplotlib.pyplot as plt
    
    dashboard_data_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
    )
    
    if dashboard_data_file.exists():
        with open(dashboard_data_file, 'r') as f:
            dashboard_data = json.load(f)
        
        top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
        
        if top_factors:
            # Extract data for plotting
            features = [f['feature'] for f in top_factors]
            importance = [f.get('causal_responsibility', 
                              f.get('importance', 
                                   f.get('combined_importance_norm', 0))) 
                        for f in top_factors]
            
            # Create plot
            plt.figure(figsize=(10, max(6, len(features) * 0.4)))
            plt.barh(range(len(features)), importance)
            plt.yticks(range(len(features)), features)
            plt.xlabel('Causal Responsibility / Importance')
            plt.title(f'Top {TOP_K} Causal Factors - {COHORT} Model (All Cohorts)')
            plt.gca().invert_yaxis()  # Top factor at top
            plt.tight_layout()
            
            # Save plot
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / f"top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved plot: {plot_file}")
            
            plt.show()
        else:
            print("\n⚠ No causal factors available for plotting")
    else:
        print(f"\n⚠ Dashboard data not found: {dashboard_data_file}")
        print("  Run SHAP/FFA analysis first (Section 4)")
            
except ImportError:
    print("\n⚠ Matplotlib not available. Skipping visualizations.")
    print("  Install with: pip install matplotlib")

## 7. Export Summary

Create a summary JSON file with all results for the Combined model.

In [ ]:
# Create workflow summary
from datetime import datetime

summary = {
    "workflow": "Calculator Model Training + SHAP/FFA Analysis",
    "model_strategy": "Single Combined model for all cohorts",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "cohort": COHORT,
        "top_k": TOP_K,
        "weight_catboost": WEIGHT_CATBOOST,
        "weight_xgboost": WEIGHT_XGBOOST,
        "debug_mode": DEBUG_MODE
    },
    "model": {}
}

# Best model
best_model_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "best_model.txt"
if best_model_file.exists():
    with open(best_model_file, 'r') as f:
        content = f.read()
        lines = content.split('\n')
        for line in lines:
            if line.startswith("Best Model:"):
                summary["model"]["best_model"] = line.replace("Best Model: ", "").strip()
            elif line.startswith("C-index:"):
                try:
                    summary["model"]["c_index"] = float(line.replace("C-index: ", "").strip())
                except:
                    pass

# Dashboard data
dashboard_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT / "dashboard_data.json"
if dashboard_file.exists():
    with open(dashboard_file, 'r') as f:
        dashboard_data = json.load(f)
        summary["model"]["top_factors_count"] = len(dashboard_data.get('top_causal_factors', []))
        if dashboard_data.get('top_causal_factors'):
            summary["model"]["top_factor"] = dashboard_data['top_causal_factors'][0]['feature']
            summary["model"]["top_factor_importance"] = dashboard_data['top_causal_factors'][0].get(
                'causal_responsibility', 
                dashboard_data['top_causal_factors'][0].get('importance', 0)
            )
        
        # List top 5 factors
        top5 = dashboard_data.get('top_causal_factors', [])[:5]
        summary["model"]["top_5_factors"] = [
            {
                "feature": f['feature'],
                "importance": f.get('causal_responsibility', 
                                  f.get('importance', 
                                       f.get('combined_importance_norm', 0)))
            }
            for f in top5
        ]

# Feature count
feature_file = CALCULATOR_DIR / "outputs" / "models" / COHORT / "feature_names.json"
if feature_file.exists():
    with open(feature_file, 'r') as f:
        features = json.load(f)
        summary["model"]["total_features"] = len(features)
        summary["model"]["key_features"] = {
            "primary_etiology": 'primary_etiology' in features or any('primary_etiology' in f for f in features),
            "vad_combined": 'vad_combined' in features,
            "vent_combined": 'vent_combined' in features,
            "ecmo_combined": 'ecmo_combined' in features,
            "donor_weight_ratio": 'donor_weight_ratio' in features,
            "donor_size_ratio": 'donor_size_ratio' in features,
            "chd_lat": 'chd_lat' in features,
            "egfr_tx": 'egfr_tx' in features,
            "txfcpra": 'txfcpra' in features,
            "hxsurg": 'hxsurg' in features
        }

# Save summary
summary_file = CALCULATOR_DIR / "outputs" / "workflow_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Workflow summary saved to: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 8. Feature Validation

Validate that all required input features are present in the trained model.

In [ ]:
# Validate features
print(f"\n{'=' * 80}")
print("Feature Validation")
print(f"{'=' * 80}")

try:
    from validate_features import validate_features
    
    print(f"\nValidating features for {COHORT} model...")
    validation_result = validate_features(COHORT)
    
    print(f"\nValidation Results:")
    print(f"  Status: {'✓ PASSED' if validation_result['status'] == 'passed' else '✗ FAILED'}")
    print(f"  Model features: {validation_result['model_feature_count']}")
    print(f"  Training features: {validation_result['training_feature_count']}")
    
    if validation_result['status'] == 'passed':
        print(f"\n  ✓ All features aligned!")
        print(f"  ✓ Model and training features match")
    else:
        print(f"\n  ⚠ Feature mismatch detected:")
        if validation_result.get('missing_in_model'):
            print(f"    Missing in model ({len(validation_result['missing_in_model'])}):")
            for feat in validation_result['missing_in_model'][:10]:
                print(f"      - {feat}")
        if validation_result.get('missing_in_training'):
            print(f"    Missing in training ({len(validation_result['missing_in_training'])}):")
            for feat in validation_result['missing_in_training'][:10]:
                print(f"      - {feat}")
    
    # Check required features
    print(f"\n  Required Features Check:")
    required_features = {
        'primary_etiology': 'primary_etiology',
        'hxsurg': 'hxsurg',
        'chd_lat': 'chd_lat',
        'hxdysdia': 'hxdysdia',
        'ecmo_combined': 'ecmo_combined',
        'vad_combined': 'vad_combined',
        'vent_combined': 'vent_combined',
        'age_txpl': 'age_txpl',
        'egfr_tx': 'egfr_tx',
        'txalt': 'txalt',
        'txast': 'txast',
        'txbili_d_r': 'txbili_d_r',
        'txbili_t_r': 'txbili_t_r',
        'txsa_r': 'txsa_r',
        'txtp_r': 'txtp_r',
        'txfcpra': 'txfcpra',
        'donisch': 'donisch',
        'donor_weight_ratio': 'donor_weight_ratio',
        'donor_size_ratio': 'donor_size_ratio'
    }
    
    model_features = set(validation_result.get('model_features', []))
    for req_name, req_var in required_features.items():
        # Check exact match or contains
        found = req_var in model_features or any(req_var in f for f in model_features)
        status = "✓" if found else "✗"
        print(f"    {status} {req_name:25s} ({req_var})")
    
except ImportError:
    print("\n⚠ validate_features module not found")
    print("  Run: python validate_features.py --cohort Combined")
except Exception as e:
    print(f"\n✗ Error validating features: {e}")
    logger.error("Error validating features", exc_info=True)

print(f"\n{'=' * 80}")